# Bengali Sentiment Analysis — Ablation Experiments

Runs the ablation chain from `thesis_plan.md`: fine-tuned BanglaBERT, class-weighted loss, FGM adversarial training, and LoRA (flagship novelty), for both the 3-class and 2-class tasks.

**Before running:** Settings (right panel) → Accelerator: GPU T4 x2, Internet: On. Push the `thesis_defance` repo to GitHub from your own machine first (`git push -u origin main`) if you haven't already — this notebook clones it.

## 0. GPU check

In [ ]:
!nvidia-smi

## 1. Clone the repo

Clones fresh, or pulls latest if this cell has already run once in this session.

In [ ]:
import os

REPO_URL = "https://github.com/habibour/thesis_defance.git"
REPO_DIR = "/kaggle/working/thesis_defance"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

In [ ]:
!cd {REPO_DIR} && echo "HEAD: $(git rev-parse --short HEAD)" && git log -1 --format="%h %ci %s"

## 2. Install dependencies

In [ ]:
!pip install -q -r {REPO_DIR}/requirements.txt
# If the BanglaBERT normalizer package fails to install here, training still
# works -- src/preprocessing.py falls back to manual normalization.

## 3. Dataset paths

Matches the Kaggle dataset mount given in the project brief.

In [ ]:
DATASET_DIR = "/kaggle/input/datasets/reversedthoutgts/bangla-dataset"
TRAIN_PATH = f"{DATASET_DIR}/train_.csv"
TEST_PATH = f"{DATASET_DIR}/test_.csv"

!ls -la {DATASET_DIR}

import pandas as pd
print("train:", pd.read_csv(TRAIN_PATH).shape)
print("test:", pd.read_csv(TEST_PATH).shape)

## 4. Sanity-check the data pipeline

In [ ]:
import sys
sys.path.insert(0, f"{REPO_DIR}/src")
from dataset import build_datasets
import numpy as np

bundle = build_datasets(train_path=TRAIN_PATH, test_path=TEST_PATH, task="3class")
print({k: len(v) for k, v in bundle.dataset_dict.items()}, bundle.num_labels, bundle.label_names)
print("test label counts:", np.bincount(bundle.dataset_dict["test"]["label"]))
# Expect [877, 843, 1280] for [Neutral, Positive, Negative] -- matches the base
# paper's Table III exactly. If this doesn't match, stop and check the CSVs.

## 5. Smoke test (1 epoch, ~5 min)

Run this before committing GPU hours to the full ablation chain -- catches config/environment errors early.

In [ ]:
!cd {REPO_DIR}/src && python train.py \
    --task 3class --model_name csebuetnlp/banglabert \
    --train_path {TRAIN_PATH} --test_path {TEST_PATH} \
    --output_dir /kaggle/working/runs \
    --epochs 1 --run_name smoke_test --seed 42

In [ ]:
# Clean up the smoke test run once it succeeds so it doesn't clutter the summary table below.
!rm -rf /kaggle/working/runs/smoke_test

## 6. Core ablation — 3-class task

~25-90 min per cell depending on config (see `thesis_plan.md` section 3 for time estimates). Run cells one at a time and check the printed `test_accuracy`/`test_macro_f1` before moving on.

In [ ]:
# Step 1: fine-tuned BanglaBERT, no imbalance handling
!cd {REPO_DIR}/src && python train.py \
    --task 3class --model_name csebuetnlp/banglabert \
    --train_path {TRAIN_PATH} --test_path {TEST_PATH} \
    --output_dir /kaggle/working/runs --seed 42

In [ ]:
# Step 2: + class-weighted loss
!cd {REPO_DIR}/src && python train.py \
    --task 3class --model_name csebuetnlp/banglabert \
    --train_path {TRAIN_PATH} --test_path {TEST_PATH} \
    --output_dir /kaggle/working/runs \
    --use_class_weights --seed 42

In [ ]:
# Step 3: + FGM adversarial training
!cd {REPO_DIR}/src && python train.py \
    --task 3class --model_name csebuetnlp/banglabert \
    --train_path {TRAIN_PATH} --test_path {TEST_PATH} \
    --output_dir /kaggle/working/runs \
    --use_class_weights --use_fgm --seed 42

## 7. Flagship — LoRA (parameter-efficient fine-tuning)

Same class weighting + FGM setup as step 3, LoRA instead of full fine-tuning. Watch the printed trainable-vs-total param counts to confirm LoRA is actually shrinking the trainable footprint.

In [ ]:
!cd {REPO_DIR}/src && python train.py \
    --task 3class --model_name csebuetnlp/banglabert \
    --train_path {TRAIN_PATH} --test_path {TEST_PATH} \
    --output_dir /kaggle/working/runs \
    --use_class_weights --use_fgm --use_lora --seed 42

Optional: LoRA's memory savings make BanglaBERT-large feasible on a single T4. Try it if the base model results leave headroom before the 75% target.

In [ ]:
!cd {REPO_DIR}/src && python train.py \
    --task 3class --model_name csebuetnlp/banglabert_large \
    --train_path {TRAIN_PATH} --test_path {TEST_PATH} \
    --output_dir /kaggle/working/runs \
    --use_class_weights --use_fgm --use_lora --seed 42

## 7b. 3-model ensemble (adapted from BLP-2023 Knowdee)

Knowdee's BLP-2023 system (F1-micro 0.7267, 2nd/30 teams) used a 10-fold BanglaBERT-large ensemble + FGM + pseudo-labeling. Adapted for our time/compute budget: a 3-model soft-vote ensemble (diversity from seed + model size, no pseudo-labeling since our test set has real gold labels -- training on it would compromise held-out evaluation). See the chat discussion in `thesis_plan.md` for the full reasoning.

All 3 runs below now save `test_probs.npy` (softmax probabilities) so they can be soft-voted -- rerun any earlier LoRA run that predates this if you want to include it, since older runs only have `test_preds.npy`.

In [ ]:
# Member 1: banglabert-base, LoRA+CW+FGM, seed 42
!cd {REPO_DIR}/src && python train.py \
    --task 3class --model_name csebuetnlp/banglabert \
    --train_path {TRAIN_PATH} --test_path {TEST_PATH} \
    --output_dir /kaggle/working/runs \
    --use_class_weights --use_fgm --use_lora --seed 42

In [ ]:
# Member 2: banglabert-base, LoRA+CW+FGM, seed 123 (diversity via seed)
!cd {REPO_DIR}/src && python train.py \
    --task 3class --model_name csebuetnlp/banglabert \
    --train_path {TRAIN_PATH} --test_path {TEST_PATH} \
    --output_dir /kaggle/working/runs \
    --use_class_weights --use_fgm --use_lora --seed 123

In [ ]:
# Member 3: banglabert-large, LoRA+CW+FGM, seed 42 (diversity via model size)
!cd {REPO_DIR}/src && python train.py \
    --task 3class --model_name csebuetnlp/banglabert_large \
    --train_path {TRAIN_PATH} --test_path {TEST_PATH} \
    --output_dir /kaggle/working/runs \
    --use_class_weights --use_fgm --use_lora --seed 42

In [ ]:
# Soft-vote the 3 members together (averages predicted-class probabilities,
# same principle as Knowdee's confidence-summing -- see evaluate.py docstring)
!cd {REPO_DIR}/src && python evaluate.py ensemble --mode soft --runs \
    /kaggle/working/runs/3class_banglabert_cw_fgm_lora_seed42 \
    /kaggle/working/runs/3class_banglabert_cw_fgm_lora_seed123 \
    /kaggle/working/runs/3class_banglabert_large_cw_fgm_lora_seed42

Compare this ensemble's accuracy/macro-F1 against each individual member's `test_metrics.json` (section 9 below pulls all of these into one table automatically). If the ensemble doesn't clear the best individual member by a meaningful margin, the added complexity likely isn't worth carrying into the thesis -- a single well-tuned model is easier to defend than an ensemble that doesn't earn its keep.

## 8. 2-class task

Repeat the same progression. Imbalance is milder here, so try without class weights first.

In [ ]:
!cd {REPO_DIR}/src && python train.py \
    --task 2class --model_name csebuetnlp/banglabert \
    --train_path {TRAIN_PATH} --test_path {TEST_PATH} \
    --output_dir /kaggle/working/runs --seed 42

In [ ]:
!cd {REPO_DIR}/src && python train.py \
    --task 2class --model_name csebuetnlp/banglabert \
    --train_path {TRAIN_PATH} --test_path {TEST_PATH} \
    --output_dir /kaggle/working/runs \
    --use_fgm --seed 42

In [ ]:
!cd {REPO_DIR}/src && python train.py \
    --task 2class --model_name csebuetnlp/banglabert \
    --train_path {TRAIN_PATH} --test_path {TEST_PATH} \
    --output_dir /kaggle/working/runs \
    --use_fgm --use_lora --seed 42

## 9. Results summary

Collects every run's `test_metrics.json` into one comparison table -- this is the source for the comparison table in `thesis_plan.md` section 5.

In [ ]:
import glob, json
import pandas as pd

rows = []
for path in glob.glob("/kaggle/working/runs/*/test_metrics.json"):
    with open(path) as f:
        m = json.load(f)
    rows.append({
        "run_name": m["run_name"],
        "task": m["task"],
        "model_name": m["model_name"],
        "class_weights": m["use_class_weights"],
        "fgm": m["use_fgm"],
        "lora": m.get("use_lora", False),
        "seed": m["seed"],
        "accuracy": round(m["test_accuracy"], 4),
        "macro_f1": round(m["test_macro_f1"], 4),
        "trainable_params": m.get("trainable_params"),
    })

summary = pd.DataFrame(rows).sort_values(["task", "macro_f1"], ascending=[True, False])
summary

## 10. Evaluation utilities

Confusion matrix, McNemar's significance test, and ensemble voting -- see `README.md` for full usage. Fill in actual run directory names from the summary table above.

**Label-shift correction (free, no retraining):** we diagnosed a real class-balance shift between the train/validation pool and the test set (Negative is ~47.6% of train/val, ~42.7% of test for 3-class). This estimates the test set's true label distribution *without using test labels* (Black-Box Shift Estimation, Lipton et al. 2018) and re-weights predictions accordingly. Costs zero GPU time -- run it on any completed run that has `val_probs.npy`/`val_labels.npy` (runs trained before this feature was added won't have these files; rerun them if you want this applied).

In [ ]:
# Example -- replace with your actual run directory once training finishes
# !cd {REPO_DIR}/src && python evaluate.py priorshift --run /kaggle/working/runs/<run_name>

In [ ]:
# Confusion matrix for a specific run
# !cd {REPO_DIR}/src && python evaluate.py confusion --metrics_json /kaggle/working/runs/<run_name>/test_metrics.json

# Is the improvement statistically significant vs. a reproduced frozen-mBERT baseline?
# !cd {REPO_DIR}/src && python evaluate.py mcnemar --run_a /kaggle/working/runs/<run_a> --run_b /kaggle/working/runs/<run_b>

# Hard-vote ensemble (fallback option if 3-class target isn't hit -- see thesis_plan.md)
# !cd {REPO_DIR}/src && python evaluate.py ensemble --runs /kaggle/working/runs/<run1> /kaggle/working/runs/<run2>

## 11. Save results back into the repo

Copies each run's metrics/predictions into `results/` inside the cloned repo (under `/kaggle/working`). Download this via Kaggle's Output panel after saving a version, then copy into your local `thesis_defance/results/` and commit/push from your own machine.

In [ ]:
import shutil, os

os.makedirs(f"{REPO_DIR}/results", exist_ok=True)
for run_dir in glob.glob("/kaggle/working/runs/*/"):
    run_name = os.path.basename(os.path.normpath(run_dir))
    dst = f"{REPO_DIR}/results/{run_name}"
    os.makedirs(dst, exist_ok=True)
    for fname in ["test_metrics.json", "test_preds.npy", "test_labels.npy"]:
        src = os.path.join(run_dir, fname)
        if os.path.exists(src):
            shutil.copy(src, dst)

summary.to_csv(f"{REPO_DIR}/results/summary.csv", index=False)
print("Copied result artifacts + summary.csv into", f"{REPO_DIR}/results")